# 🚀 ACE-Net Master Baseline Model Training & Evaluation
### End-to-End Multimodal Deepfake Consistency Training on 14k Preprocessed Dataset

### 🌟 Workflow Overview:
1. **Load Preprocessed Tensors:** Directly streams `.npy` and `.jpg` features from Google Drive.
2. **Load Pretrained Stage-2 Backbone:** Uses `stage2_acenet.pt` from inyong `checkpoints/` folder.
3. **Train on 14,588 Clips (`final_train_manifest.csv`):** Trains with BCE Loss, AdamW, and Cosine Annealing.
4. **Validate Per Epoch (`final_val_manifest.csv`):** Evaluates Accuracy, AUC, and F1 at every epoch and auto-saves the **`best_baseline_model.pth`** to Google Drive!
5. **Final Testing (`final_test_manifest.csv`):** Produces the official Baseline Results Table for your thesis!

## Step 1: Connect to T4 GPU & Mount Google Drive

In [ ]:
from google.colab import drive
import os, sys, torch

drive.mount('/content/drive')
print('GPU Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device Name  :', torch.cuda.get_device_name(0))
else:
    print('⚠️ WARNING: GPU is not enabled! Go to Runtime > Change runtime type > T4 GPU!')

## Step 2: Clone Baseline Repository & Checkout Active Branch

In [ ]:
%cd /content
!rm -rf Baseline_Training
!git clone https://github.com/gjvlio/Baseline_Training.git
%cd Baseline_Training
!git checkout feat/baseline-preprocessing-jc
!git pull
!git log --oneline -1

## Step 3: Install Core Dependencies

In [ ]:
!pip install -q scikit-learn transformers
print('✅ Dependencies installed successfully!')

## Step 4: Run Stage-2 Baseline Training & Evaluation (1-Click Run)

In [ ]:
import os
from pathlib import Path

DRIVE_BASE = Path('/content/drive/MyDrive/THESIS_MOTHERFILE/Baseline_training')
PREPROCESSED_ROOT = DRIVE_BASE / 'Baseline preprocessed'
MANIFEST_DIR = Path('/content/Baseline_Training/Manifests/final_manifest_jc')

# Checkpoint paths
STAGE2_CKPT = DRIVE_BASE / 'checkpoints' / 'stage2_acenet.pt'
OUTPUT_MODEL_DIR = DRIVE_BASE / 'checkpoints'

TRAIN_CSV = MANIFEST_DIR / 'final_train_manifest.csv'
VAL_CSV = MANIFEST_DIR / 'final_val_manifest.csv'
TEST_CSV = MANIFEST_DIR / 'final_test_manifest.csv'

print('=' * 75)
print('Train Manifest :', TRAIN_CSV.exists())
print('Val Manifest   :', VAL_CSV.exists())
print('Test Manifest  :', TEST_CSV.exists())
print('Preprocessed   :', PREPROCESSED_ROOT.exists())
print('Stage-2 Ckpt   :', STAGE2_CKPT.exists())
print('=' * 75)

ckpt_arg = f"--ckpt '{STAGE2_CKPT}'" if STAGE2_CKPT.exists() else ""

!python -m src.train_baseline_engine \
    --train-manifest '{TRAIN_CSV}' \
    --val-manifest '{VAL_CSV}' \
    --test-manifest '{TEST_CSV}' \
    --preprocessed-root '{PREPROCESSED_ROOT}' \
    {ckpt_arg} \
    --output-dir '{OUTPUT_MODEL_DIR}' \
    --batch-size 32 \
    --epochs 20 \
    --lr 1e-4 \
    --num-workers 2 \
    --freeze-backbones \
    --device cuda